# 03 · Rate laws and stoichiometry

A rate law says how fast; stoichiometry says what everything else is doing while A reacts. Between them
they turn a design equation in $X$ into something solvable: the rate as a function of conversion. This
notebook builds stoichiometric tables for liquids and for gases (where the volume changes), handles the
expansion factor, reversible reactions and the equilibrium limit, and shows how temperature enters through
Arrhenius and van 't Hoff.

**What you will learn**
- build stoichiometric tables for liquids and gases
- use the expansion factor with pressure and temperature
- compute equilibrium conversions for reversible reactions
- see how Arrhenius and van 't Hoff combine

**Before you start:** notebook 01  ·  **Time:** about 60 min

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engreact import kinetics, thermo
from engreact.kinetics import Reaction
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## The stoichiometric table

In [2]:
rxn = Reaction({"A": -1, "B": -2, "C": 1}, key="A")
print("reaction:", rxn, "| moles change per mole of A:", rxn.delta)
feed = {"A": 1.0, "B": 3.0, "I": 0.5}                                  # Theta_B = 3, inert
X = np.array([0, 0.25, 0.5, 0.75, 1.0])
tab = kinetics.stoichiometric_table(rxn, feed, X, phase="liquid", C_A0=2.0)
print("\nliquid phase (C_A0 = 2 mol/L):")
print(pd.DataFrame({"X": X, **{s: tab["C"][s] for s in ("A", "B", "C", "I")}}).round(3).to_string(index=False))

reaction: A + 2 B -> C | moles change per mole of A: -2.0

liquid phase (C_A0 = 2 mol/L):
   X   A   B   C   I
0.00 2.0 6.0 0.0 1.0
0.25 1.5 5.0 0.5 1.0
0.50 1.0 4.0 1.0 1.0
0.75 0.5 3.0 1.5 1.0
1.00 0.0 2.0 2.0 1.0


Every concentration is $C_{A0}(\Theta_i + \nu_i X/|\nu_A|)$: B with $\Theta_B = 3$ falls twice as fast as A
because two are consumed per A, C rises, the inert stays. The table is the bookkeeping that lets a rate law in
concentrations become a function of $X$ alone.

## Gas phase: the volume changes

In [3]:
gas = Reaction({"A": -1, "B": 2}, key="A")                       # A -> 2B: moles double
for inert in (0.0, 1.0, 4.0):
    feed = {"A": 1.0, "I": inert}
    tab = kinetics.stoichiometric_table(gas, feed, X, phase="gas", C_A0=1.0)
    print(f"y_A0 = {1/(1+inert):.2f}: epsilon = {tab['epsilon']:.2f};  C_A(X) = {np.round(tab['C']['A'], 3)}")
print("\ncompare the liquid-like case (no expansion): C_A(X) =", np.round(1.0 * (1 - X), 3))

y_A0 = 1.00: epsilon = 1.00;  C_A(X) = [1.    0.6   0.333 0.143 0.   ]
y_A0 = 0.50: epsilon = 0.50;  C_A(X) = [1.    0.667 0.4   0.182 0.   ]
y_A0 = 0.20: epsilon = 0.20;  C_A(X) = [1.    0.714 0.455 0.217 0.   ]

compare the liquid-like case (no expansion): C_A(X) = [1.   0.75 0.5  0.25 0.  ]


When moles increase, the gas expands, and every concentration is divided by $1 + \varepsilon X$: at complete
conversion of a pure A -> 2B feed the volume has doubled, so $C_A$ falls faster than $(1 - X)$ and the reaction
slows itself. Diluting with inert reduces $\varepsilon$ - one reason gas-phase reactors run with inerts or in
excess of one reactant.

## Pressure, temperature and the rate as a function of X

In [4]:
k = kinetics.arrhenius(5.0e7, 68e3, 320.0)
C_A0 = 40.0                                                            # mol/m3, a gas at 1 bar and 300 K-ish
for eps, label in ((0.0, "no expansion"), (1.0, "A -> 2B, pure feed")):
    C_A = kinetics.concentration_from_conversion(C_A0, X, eps=eps)
    print(f"{label:<20}: -r_A(X) = k C_A = {np.round(k * C_A, 5)} mol/(m3 s)")
print("\nwith 20 % pressure drop and the gas heated from T0 to 1.1 T0 at X = 0.5:",
      f"C_A = {kinetics.concentration_from_conversion(C_A0, 0.5, eps=1.0, P_ratio=0.8, T_ratio=1.1):.2f} mol/m3 (vs {kinetics.concentration_from_conversion(C_A0, 0.5, eps=1.0):.2f} at P0, T0)")

no expansion        : -r_A(X) = k C_A = [0.0159  0.01192 0.00795 0.00397 0.     ] mol/(m3 s)
A -> 2B, pure feed  : -r_A(X) = k C_A = [0.0159  0.00954 0.0053  0.00227 0.     ] mol/(m3 s)

with 20 % pressure drop and the gas heated from T0 to 1.1 T0 at X = 0.5: C_A = 9.70 mol/m3 (vs 13.33 at P0, T0)


The full gas-phase expression $C_A = C_{A0}\frac{1 - X}{1 + \varepsilon X}\frac{P}{P_0}\frac{T_0}{T}$ collects
three effects: consumption, expansion and the state of the gas. Pressure drop (notebook 04) and heating
(notebook 07) both enter the rate through it.

## Reversible reactions and equilibrium conversion

In [5]:
rev = Reaction({"A": -1, "B": 1}, key="A")
for Kc in (0.5, 2.0, 10.0, 100.0):
    print(f"A <-> B with Kc = {Kc:5.1f}: X_e = {kinetics.equilibrium_conversion(rev, {'A': 1.0}, Kc, C_A0=1.0):.3f}")
diss = Reaction({"A": -1, "B": 2}, key="A")
for Kc in (0.1, 1.0, 10.0):
    Xe_l = kinetics.equilibrium_conversion(diss, {"A": 1.0}, Kc, C_A0=1.0, phase="liquid")
    Xe_g = kinetics.equilibrium_conversion(diss, {"A": 1.0}, Kc, C_A0=1.0, phase="gas")
    print(f"A <-> 2B with Kc = {Kc:5.1f} (C_A0 = 1): X_e = {Xe_l:.3f} at constant volume, {Xe_g:.3f} with gas expansion")
C = kinetics.stoichiometric_table(rev, {"A": 1.0}, 0.3, "liquid", 1.0)["C"]
print(f"\nnet rate of A <-> B (kf = 0.1, Kc = 2) at X = 0.3: {kinetics.reversible_rate(0.1, 2.0, [C['A']], [C['B']]):.4f}; at X = X_e: {kinetics.reversible_rate(0.1, 2.0, [1/3], [2/3]):.2e}")

A <-> B with Kc =   0.5: X_e = 0.333
A <-> B with Kc =   2.0: X_e = 0.667
A <-> B with Kc =  10.0: X_e = 0.909
A <-> B with Kc = 100.0: X_e = 0.990
A <-> 2B with Kc =   0.1 (C_A0 = 1): X_e = 0.146 at constant volume, 0.156 with gas expansion
A <-> 2B with Kc =   1.0 (C_A0 = 1): X_e = 0.390 at constant volume, 0.447 with gas expansion
A <-> 2B with Kc =  10.0 (C_A0 = 1): X_e = 0.766 at constant volume, 0.845 with gas expansion

net rate of A <-> B (kf = 0.1, Kc = 2) at X = 0.3: 0.0550; at X = X_e: 0.00e+00


A reversible reaction stops at the equilibrium conversion, where the forward and reverse rates cancel; the
net rate law $k_f(C_A - C_B/K_c)$ builds that in. Expansion helps a dissociation (Le Chatelier: more moles,
so lower pressure favours products), which the gas-phase table captures automatically.

## Temperature: Arrhenius and van 't Hoff together

In [6]:
stoich = {"C2H4": -1, "H2O": -1, "C2H5OH": 1}                       # ethylene hydration, exothermic
for T in (400.0, 500.0, 600.0):
    K = thermo.equilibrium_constant(stoich, T)
    Kc = thermo.kc_from_kp(K, -1, T)
    print(f"{T:.0f} K: Delta H = {thermo.heat_of_reaction(stoich, T)/1e3:6.1f} kJ/mol, Kp = {K:9.3g} 1/bar, Kc = {Kc:9.3g} m3/mol")
print("\nrate constants rise with T (Arrhenius) but the equilibrium falls (exothermic): the optimum is a compromise - notebook 07")

400 K: Delta H =  -46.5 kJ/mol, Kp =     0.226 1/bar, Kc =   0.00753 m3/mol
500 K: Delta H =  -47.1 kJ/mol, Kp =    0.0135 1/bar, Kc =  0.000563 m3/mol
600 K: Delta H =  -47.3 kJ/mol, Kp =   0.00204 1/bar, Kc =  0.000102 m3/mol

rate constants rise with T (Arrhenius) but the equilibrium falls (exothermic): the optimum is a compromise - notebook 07


For an exothermic reversible reaction, heating speeds the approach to equilibrium and lowers the
equilibrium itself. Ethylene hydration, ammonia synthesis and SO$_2$ oxidation all live on this trade-off,
and the reactor's temperature profile is designed around it.

## Inside the algorithm: the gas-phase concentration

In [7]:
X_, eps_, C0_ = 0.5, 1.0, 40.0
print("by hand:", C0_ * (1 - X_) / (1 + eps_ * X_), " library:", float(kinetics.concentration_from_conversion(C0_, X_, eps_)))

by hand: 13.333333333333334  library: 13.333333333333334


## Exercises
1. Ammonia synthesis N2 + 3H2 <-> 2NH3 with a stoichiometric feed at 450 degC: compute Kp from `thermo`, convert to Kc at 100 bar, and find the equilibrium conversion (gas phase, epsilon = -0.5). Then repeat at 300 bar and at 400 degC to see why the process runs at high pressure.
2. For A + B -> C in the liquid phase with C_A0 = 1 mol/L, compare the rate law -r = k C_A C_B as a function of X for Theta_B = 1, 2 and 5. Explain why excess B makes the reaction look first order.
3. **Implement it yourself:** write the stoichiometric table for 2A + B -> 3C in the gas phase with a 2:1:1 feed of A:B:inert by hand (Theta, nu, epsilon) and check every column against `kinetics.stoichiometric_table` at X = 0.4.